In [0]:
from pyspark.sql.functions import *

In [0]:
customers = spark.read.format("delta").load(
    "/Volumes/workspace/default/fmcg_data/silver/customers"
)

products = spark.read.format("delta").load(
    "/Volumes/workspace/default/fmcg_data/silver/products"
)

stores = spark.read.format("delta").load(
    "/Volumes/workspace/default/fmcg_data/silver/stores"
)

sales = spark.read.format("delta").load(
    "/Volumes/workspace/default/fmcg_data/silver/sales"
)

In [0]:
customers.show(5)
products.show(5)
stores.show(5)
sales.show(5)

+-----------+-----------------+-------+-----------+------------+--------------------+-------------------+--------------------+
|customer_id|    customer_name|   city|      state|       phone|               email|        source_file|      ingestion_time|
+-----------+-----------------+-------+-----------+------------+--------------------+-------------------+--------------------+
|         12|Abhiram Bhatnagar| Mumbai|Maharashtra|  6504150263|  gmurty@example.net|customers_dirty.csv|2026-07-09 12:40:...|
|         18|      Gauri Bassi| Mumbai|Maharashtra|917742007745|   uvyas@example.com|customers_dirty.csv|2026-07-09 12:40:...|
|         38|      Akshay Kaul|Chennai| Tamil Nadu|  9495435766|aryanagrawal@exam...|customers_dirty.csv|2026-07-09 12:40:...|
|         67|   Shivansh Baria|   Pune|Maharashtra|916656219116|regechatresh@exam...|customers_dirty.csv|2026-07-09 12:40:...|
|         70|      Hemani Bora|Chennai| Tamil Nadu|  8556949874| vpandey@example.net|customers_dirty.csv|2026-0

In [0]:
dim_customer = customers.select(
    "customer_id",
    "customer_name",
    "city",
    "state",
    "email"
)

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/dim_customer")

In [0]:
dim_product = products.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "price"
)

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/dim_product")

In [0]:
dim_store = stores.select(
    "store_id",
    "store_name",
    "city",
    "state"
)

dim_store.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/dim_store")

In [0]:
dim_date = sales.select("sale_date").distinct()

dim_date = dim_date \
    .withColumn("year", year("sale_date")) \
    .withColumn("month", month("sale_date")) \
    .withColumn("day", dayofmonth("sale_date")) \
    .withColumn("quarter", quarter("sale_date")) \
    .orderBy("sale_date")

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/dim_date")

In [0]:
fact_sales = sales.select(
    "sale_id",
    "customer_id",
    "product_id",
    "store_id",
    "sale_date",
    "quantity",
    "amount"
)

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/fact_sales")

In [0]:
sales_by_state = fact_sales.join(
    dim_store,
    "store_id"
).groupBy("state").agg(
    round(sum("amount"), 2).alias("total_sales")
).orderBy(desc("total_sales"))

sales_by_state.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/sales_by_state")

sales_by_state.show()

+-------------+-----------+
|        state|total_sales|
+-------------+-----------+
|  Maharashtra| 4525927.08|
|Uttar Pradesh| 2217779.74|
|  West Bengal| 2054720.56|
|    Karnataka| 1608395.28|
|   Tamil Nadu| 1559102.71|
|    Telangana| 1140680.62|
|        Delhi| 1102175.06|
+-------------+-----------+



In [0]:
top_products = fact_sales.join(
    dim_product,
    "product_id"
).groupBy(
    "product_name",
    "category"
).agg(
    round(sum("amount"), 2).alias("revenue")
).orderBy(desc("revenue"))

top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/top_products")

top_products.show(10)

+-------------+-------------+---------+
| product_name|     category|  revenue|
+-------------+-------------+---------+
|     Mollitia|Personal Care|171543.69|
|     Adipisci|        Dairy|115165.94|
|          Nam|    Beverages|103177.35|
|        Porro|        Dairy| 95295.84|
|Reprehenderit|Personal Care| 90625.07|
|        Nihil|    Household| 88501.86|
|  Dignissimos|    Household| 86972.76|
|     Officiis|Personal Care| 83495.56|
|        Autem|        Dairy|  82341.7|
|         Quae|        Dairy| 81930.08|
+-------------+-------------+---------+
only showing top 10 rows


In [0]:
monthly_sales = fact_sales \
    .withColumn("year", year("sale_date")) \
    .withColumn("month", month("sale_date")) \
    .groupBy("year", "month") \
    .agg(
        round(sum("amount"), 2).alias("total_sales")
    ) \
    .orderBy("year", "month")

monthly_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/fmcg_data/gold/monthly_sales")

monthly_sales.show()

+----+-----+-----------+
|year|month|total_sales|
+----+-----+-----------+
|2024|    7|   480883.3|
|2024|    8|  601789.96|
|2024|    9|  579766.32|
|2024|   10|   617554.3|
|2024|   11|  648129.34|
|2024|   12|  599906.12|
|2025|    1|  614512.27|
|2025|    2|  552917.31|
|2025|    3|  575901.52|
|2025|    4|  588270.34|
|2025|    5|  611229.63|
|2025|    6|   540602.0|
|2025|    7|  561234.14|
|2025|    8|  698703.09|
|2025|    9|  571032.95|
|2025|   10|  583235.04|
|2025|   11|  536349.37|
|2025|   12|  609759.48|
|2026|    1|  613835.58|
|2026|    2|  554342.56|
+----+-----+-----------+
only showing top 20 rows


In [0]:
dim_customer.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_customer")

dim_product.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_product")

dim_store.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_store")

dim_date.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_date")

fact_sales.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.fact_sales")

print("✅ Gold tables registered successfully!")

✅ Gold tables registered successfully!
